# H1 V1 · U-Net counting bypass for Building Damage Counting (KR2, no VLM)

Task **H1**, variant **V1**, from `Essential_task_delegation/Next_Steps_Roadmap.md`. Goal **O2 / KR2**: beat the base model's **34.2%** on BDC (our fine-tuned 7B scored 23.58%; the old 499A hybrid 13.11%).

**Why this can work (checked on 18 Oct 2026)**
- DisasterM3's BDC answers are **8-connected blob counts of its own class masks**. Counting blobs in DisasterM3's validation masks reproduces all 1,055 training-split BDC answers on our 645 validation images: intact 555/555, damaged 284/284, destroyed 216/216.
- So counting blobs in R9's predicted mask uses the same definition as the answer key. A perfect segmenter would score 100%.
- **The classes:** Intact (2,671 questions), Damaged (1,345) and Totally-destroyed (966). There are no "total" questions.
- **The options:** the nearest wrong option is about 20% away from the true count for intact (median), so a count must land within about ±10%.
- **SAR:** 1,399 questions use a SAR after-image. Each has an optical twin of the same tile in the benchmark, which the U-Net reads instead. SAR is scored separately.

**First run (18 Oct): the tuning worked, but the benchmark part found no images and scored 0%.**
- **Tuning:** R9's counts answer 31.4% of the intact, 40.1% of the damaged and 42.1% of the destroyed validation questions (about 36% overall).
- **Benchmark:** `benchmark_release.json` was read from `disasterm3-mirror/DisasterM3_Bench`, which keeps the images elsewhere.
- **Fix:** Cell 4 now indexes every PNG under any `test_images` folder in `/kaggle/input` by file name, and stops if fewer than 99% of the images are found. The saved outputs below are from that first run until it's run again.

**What it does**
1. **Tunes the counter** on those 1,055 validation-image questions: R9 never trained on these images, and they aren't benchmark questions. Per class it picks the standard or TTA mask, an optional 3×3 closing, and the minimum blob area.
2. **Segments** the 2,655 distinct after-images used by the six benchmark tasks (R9 + 8-view TTA), counts blobs, and answers all 4,982 BDC questions with the nearest option.
3. **Scores** them overall, by class, and by sensor (optical, or SAR via the optical twin), and decides **Gate G4** (V1 > 34.2%).
4. **Saves, for V2, H2, T2 and T3:**
   - `facts.json`: per image, the counts and where the damage is;
   - `masks/` (R9 TTA, 0–3) for every after-image.

**How to run on Kaggle**
1. Settings: GPU T4 ×1, internet ON.
2. Inputs:
   - the DisasterM3 mirror (attached automatically; it supplies the validation images);
   - **`redwannewaz/disasterm3-bench-mirror-v2`** (`benchmark_release.json` and `test_images/`). Images are found by file name in any attached dataset.
3. Secret `HF_TOKEN` (R9's private repo), as in E1. Or attach a dataset with `R9/final_model.pth` (from `Champion_Task/R9-results/d4_R9/`).
4. Run with `SMOKE_TEST = True` first (20 validation and 40 benchmark images). Then use Save & Run All (about 30 minutes: the tuning half took 6).
5. Outputs in `/kaggle/working/H1/`:
   - `H1_v1_summary.csv`, `H1_v1_tuning.csv`, `H1_v1_per_question.csv`;
   - `results/building_damage_counting/R9-unet-counting/finished.jsonl`, which `4_score_results.py` reads;
   - `facts.json`, `masks/`, `config.json`, `README.md`.

   Publish the output as the Kaggle dataset `h1-v1-facts` for V2.

**Done when:** V1's BDC accuracy is reported with its breakdown, and the Gate G4 line is printed.

In [1]:
# ── Cell 1: Environment Setup (Run-All safe, identical to the D4 notebooks) ──
# Kaggle's image moved to Python 3.13, which has no torch 2.4.1 wheel. Use Kaggle's own
# torch/torchvision/opencv and pin only the libraries whose API this notebook depends on.
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "segmentation-models-pytorch": "0.3.4",   # baseline version (checkpoint key layout)
    "albumentations": "1.4.21",               # 1.x API (PadIfNeeded value=/mask_value=)
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *(f"{k}=={v}" for k, v in PINS.items()), "huggingface_hub[hf_xet]"],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        # background "Save & Run All": nothing imported yet, so new versions load on first import
        print("✓ Dependencies installed (background run) — continuing without restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed — proceeding.")

import torch
print(f"  torch {torch.__version__}, python {sys.version.split()[0]}")

⏳ Installing pinned versions:
   segmentation-models-pytorch: not installed → 0.3.4
   albumentations: 2.0.8 → 1.4.21
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.9 MB/s eta 0:00:00
✓ Dependencies installed (background run) — continuing without restart.
  torch 2.11.0+cu128, python 3.13.15


In [2]:
# ── Cell 2: Configuration ──────────────────────────────────────────────────
import glob
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/H1")

NUM_CLASSES = 4                      # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
SMOKE_TEST = False                   # True = 20 validation and 40 benchmark images
MODEL_NAME = "R9"                    # the D4 champion recipe
HF_USER = "AbrarAlam"                # checkpoint: {HF_USER}/disasterm3-d4-R9/final_model.pth
CKPT_PATH = None                     # set a path to skip the search, e.g. "/kaggle/input/my-ckpts/R9/final_model.pth"
MIN_AREAS = [0, 4, 8, 16, 24, 32, 48, 64, 96]   # tuning grid: blobs smaller than this (px) are not counted
CLOSINGS = [0, 3]                    # tuning grid: 0 = raw mask, 3 = 3x3 closing before counting
BASE_BDC, FT_BDC = 34.2, 23.58       # base Qwen2.5-VL-7B (Gate G4) and our fine-tuned 7B, BDC accuracy %

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_checkpoint(name):
    """Order: CKPT_PATH, then a /kaggle/input folder named R9 / d4_R9 / R9-results, then Hugging Face."""
    if CKPT_PATH:
        return CKPT_PATH
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo, errors = f"{HF_USER}/disasterm3-d4-{name}", []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}.")

bench_hits = sorted(glob.glob("/kaggle/input/**/benchmark_release.json", recursive=True))
if not bench_hits:
    raise FileNotFoundError("Attach redwannewaz/disasterm3-bench-mirror-v2 (benchmark_release.json and test_images/).")
BENCH_ROOT = Path(bench_hits[0]).parent
CKPT = find_checkpoint(MODEL_NAME)
(OUT_DIR / "masks").mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ H1 V1{'  [SMOKE TEST]' if SMOKE_TEST else ''}: {MODEL_NAME} from {CKPT}")
print(f"  Benchmark: {BENCH_ROOT}")
print(f"  Device:    {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ H1 V1: R9 from /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R9/snapshots/34897e1aa70742cac62a379561e1f969dccd4e10/final_model.pth
  Benchmark: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Bench
  Device:    Tesla T4


In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from the D4 notebooks) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [4]:
# ── Cell 4: Validation split, its training-split BDC questions, and the benchmark questions ──
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

NORMALIZE = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
val_transform = A.Compose([NORMALIZE, ToTensorV2()])   # E-protocol: full native image, no resize

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)

CLASS_OF = {"Intact building counting.": 1, "Damaged building counting.": 2,
            "Totally-destroyed building counting.": 3}
CLASS_NAME = {1: "intact", 2: "damaged", 3: "destroyed"}

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

# Tuning set: training-split BDC questions on the 645 validation images (R9 never trained on them)
val_names = {Path(p["image_path"]).name for p in val_pairs}
tune_qs = [e for e in raw_data if e.get("task") == "Building Damage Counting" and name_of(e["post_image_path"]) in val_names]

with open(BENCH_ROOT / "benchmark_release.json", encoding="utf-8") as f:
    bench = json.load(f)
bdc = [e for e in bench if e.get("task") == "Building Damage Counting"]
for i, e in enumerate(bdc):                      # ids as in the 499A scorer: subset name + index in benchmark order
    e["id"] = f"building_damage_counting_{i}"
    e["image"] = name_of(e["post_image_path"]).replace("_sar", "")   # SAR questions use the optical twin
SIX_TASKS = {"Disaster Bearing Bodies Recognition", "Building Damage Counting", "Disaster Type Recognition",
             "Road Damage Counting", "Disaster Scene Recognition", "Relational Reasoning"}
bench_images = sorted({name_of(e["post_image_path"]).replace("_sar", "")
                       for e in bench if e.get("task") in SIX_TASKS and e.get("post_image_path")})

if SMOKE_TEST:
    val_pairs = val_pairs[:20]
    val_names = {Path(p["image_path"]).name for p in val_pairs}
    tune_qs = [e for e in tune_qs if name_of(e["post_image_path"]) in val_names]
    bench_images = bench_images[:40]
    bdc = [e for e in bdc if e["image"] in set(bench_images)]

# Benchmark images, indexed by file name wherever the attached datasets keep them: mirrors nest them
# differently (the first run looked only in disasterm3-mirror/DisasterM3_Bench, found none and scored 0%)
bench_index = {}
for p in glob.glob("/kaggle/input/**/test_images/**/*.png", recursive=True):
    bench_index.setdefault(Path(p).name, p)

def bench_path(name):
    return bench_index.get(name)

found = sum(name in bench_index for name in bench_images)
if found < 0.99 * len(bench_images):
    raise FileNotFoundError(
        f"Only {found:,} of {len(bench_images):,} benchmark after-images were found under /kaggle/input. "
        "Attach redwannewaz/disasterm3-bench-mirror-v2 (its test_images/ folder) and run again.")

print(f"✓ Validation images: {len(val_pairs)}; tuning questions on them: {len(tune_qs)} "
      f"(expected 1,055 when SMOKE_TEST is False)")
print(f"  Benchmark BDC questions: {len(bdc):,} ({sum(e['post_image_type'] == 'SAR' for e in bdc):,} SAR); "
      f"after-images to segment: {len(bench_images):,}, found: {found:,}")
print(f"  e.g. {bench_index.get(bench_images[0])}")
print(f"  By class: {Counter(e['cls_description'] for e in bdc)}")

/usr/local/lib/python3.13/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


✓ Validation images: 645; tuning questions on them: 1055 (expected 1,055 when SMOKE_TEST is False)
  Benchmark BDC questions: 4,982 (1,399 SAR); after-images to segment: 2,655, found: 2,655
  e.g. /kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2/test_images/bata_explosion_post_0.png
  By class: Counter({'Intact building counting.': 2671, 'Damaged building counting.': 1345, 'Totally-destroyed building counting.': 966})


In [5]:
# ── Cell 5: R9 with 8-view TTA (as in E1) and the blob counter ──────────────
import segmentation_models_pytorch as smp
import torch.nn.functional as F

def load_model(path):
    """D4 checkpoints: U-Net + ResNet-50, 4 classes. Accepts a state dict or a resume checkpoint."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

def _forward_once(model, x):
    """Logits at x's own HxW: pad to a multiple of 32, exactly as in the D4 notebooks."""
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    return model(x)[..., :h, :w].float()

TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]   # the D4 notebooks' 8 dihedral views

def _tta_fwd(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x

def _tta_inv(y, k, flip):
    if flip:
        y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))

@torch.no_grad()
def predict_std_and_tta(model, x):
    """One pass over the 8 views. The identity view (k=0, no flip) is the standard prediction."""
    prob_sum, pred_std = 0, None
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        for k, flip in TTA_VIEWS:
            logits = _forward_once(model, _tta_fwd(x, k, flip))
            if k == 0 and not flip:
                pred_std = logits.argmax(1)
            prob_sum = prob_sum + torch.softmax(_tta_inv(logits, k, flip), dim=1)
    pred_tta = (prob_sum / len(TTA_VIEWS)).argmax(1)
    return (pred_std[0].cpu().numpy().astype(np.uint8),
            pred_tta[0].cpu().numpy().astype(np.uint8))

def segment(path):
    image = cv2.imread(str(path))
    if image is None:
        return None
    x = val_transform(image=cv2.cvtColor(image, cv2.COLOR_BGR2RGB))["image"].unsqueeze(0).to(DEVICE)
    return predict_std_and_tta(model, x)

def blob_areas(binary, closing=0):
    """Areas of the 8-connected blobs, the same rule DisasterM3 used for its BDC answers."""
    m = binary.astype(np.uint8)
    if closing:
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((closing, closing), np.uint8))
    return cv2.connectedComponentsWithStats(m, connectivity=8)[2][1:, cv2.CC_STAT_AREA]

def image_stats(pred_std, pred_tta):
    """Blob areas per (variant, closing, class): enough to count with any minimum area later."""
    return {(v, k, c): blob_areas(p == c, k) for v, p in (("std", pred_std), ("tta", pred_tta))
            for k in CLOSINGS for c in CLASS_NAME}

def count(stats, rule, cls):
    variant, closing, min_area = rule
    return int((stats[(variant, closing, cls)] >= min_area).sum())

def nearest_option(n, options):
    return "ABCDE"[min(range(len(options)), key=lambda i: abs(options[i] - n))]

# Self-checks: diagonal touch is one blob (8-connected); the nearest option wins
_m = np.zeros((6, 6), np.uint8)
_m[1, 1] = _m[2, 2] = 1
_m[4, 4:6] = 1
assert len(blob_areas(_m == 1)) == 2 and nearest_option(52, [10, 40, 60, 80, 100]) == "C"

model = load_model(CKPT)
print(f"✓ {MODEL_NAME} loaded; counter self-checks passed")

✓ R9 loaded; counter self-checks passed


## 1. Tune the counter on the validation images (never on the benchmark)
For each class it tries every combination of:
- prediction: standard or TTA;
- closing: none or 3×3;
- minimum blob area: 0–96 px.

It keeps the one that answers the most training-split questions on the 645 validation images. A sanity check comes first: DisasterM3's own masks must answer all of them.

In [6]:
# ── Cell 6: Segment the validation images and tune the counter on their training questions ──
import pandas as pd
from tqdm.auto import tqdm

t_start = time.time()
val_stats, gt_stats = {}, {}
for p in tqdm(val_pairs, desc="Validation images"):
    name = Path(p["image_path"]).name
    preds = segment(p["image_path"])
    gt = cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE)
    if preds is None or gt is None:
        continue
    val_stats[name] = image_stats(*preds)
    gt_stats[name] = {c: blob_areas(gt == c) for c in CLASS_NAME}
tune_qs = [e for e in tune_qs if name_of(e["post_image_path"]) in val_stats]

def accuracy(questions, answer):
    return sum(answer(e) == e["ground_truth_option"] for e in questions) / max(len(questions), 1)

# Sanity check: DisasterM3's own masks answer every tuning question (blob counts are the answer key)
gt_acc = accuracy(tune_qs, lambda e: nearest_option(
    len(gt_stats[name_of(e["post_image_path"])][CLASS_OF[e["cls_description"]]]), e["options_list"]))
print(f"Ground-truth masks answer {gt_acc:.1%} of the {len(tune_qs)} tuning questions (expected 100%)")

rows = []
for cls, cname in CLASS_NAME.items():
    qs = [e for e in tune_qs if CLASS_OF[e["cls_description"]] == cls]
    for variant in ("std", "tta"):
        for closing in CLOSINGS:
            for min_area in MIN_AREAS:
                rule = (variant, closing, min_area)
                acc = accuracy(qs, lambda e: nearest_option(
                    count(val_stats[name_of(e["post_image_path"])], rule, cls), e["options_list"]))
                rows.append({"class": cname, "variant": variant, "closing": closing, "min_area": min_area,
                             "questions": len(qs), "accuracy": round(acc, 4)})
tuning = pd.DataFrame(rows)
tuning.to_csv(OUT_DIR / "H1_v1_tuning.csv", index=False)

best = tuning.loc[tuning.groupby("class", sort=False)["accuracy"].idxmax()]
BEST_RULE = {cls: tuple(best.set_index("class").loc[cname, ["variant", "closing", "min_area"]])
             for cls, cname in CLASS_NAME.items()}
BEST_RULE = {cls: (v, int(k), int(a)) for cls, (v, k, a) in BEST_RULE.items()}
plain = tuning[(tuning["variant"] == "tta") & (tuning["closing"] == 0) & (tuning["min_area"] == 0)]
print("\nBest rule per class (tuning accuracy; optimistic, since it is chosen on these questions):")
print(best.to_string(index=False))
print("\nPlain rule for comparison (TTA, no closing, every blob):")
print(plain[["class", "questions", "accuracy"]].to_string(index=False))
print(f"\n({(time.time() - t_start) / 60:.1f} min so far)")

Validation images:   0%|          | 0/645 [00:00<?, ?it/s]

Ground-truth masks answer 100.0% of the 1055 tuning questions (expected 100%)

Best rule per class (tuning accuracy; optimistic, since it is chosen on these questions):
    class variant  closing  min_area  questions  accuracy
   intact     std        0        24        555    0.3135
  damaged     std        0         0        284    0.4014
destroyed     std        0         0        216    0.4213

Plain rule for comparison (TTA, no closing, every blob):
    class  questions  accuracy
   intact        555    0.3135
  damaged        284    0.3345
destroyed        216    0.3796

(5.7 min so far)


## 2. The benchmark: segment, count, answer
Every after-image used by the six benchmark tasks is segmented once (2,655 images, R9 + TTA). SAR questions use the optical twin of the same tile. Each image gets a fact line for V2, for example "U-Net analysis: 51 intact, 3 damaged and 0 destroyed buildings; most damage in the north-east of the image." (top of the image = north). "Most damage" is the third of a 3×3 grid with the most damaged or destroyed pixels.

In [7]:
# ── Cell 7: Segment the benchmark after-images, count, and write facts and masks ──
def compass(x, y, w, h):
    """Where a point sits in the image (top = north): 'north-east', 'west', 'centre', ..."""
    row = ("north", "", "south")[min(int(3 * y / h), 2)]
    col = ("west", "", "east")[min(int(3 * x / w), 2)]
    return "-".join(p for p in (row, col) if p) or "centre"

def most_damage_where(damage):
    """The cell of a 3x3 grid with the most damaged or destroyed pixels, as a compass word (None if none)."""
    h, w = damage.shape
    cells = [(int(damage[i * h // 3:(i + 1) * h // 3, j * w // 3:(j + 1) * w // 3].sum()), i, j)
             for i in range(3) for j in range(3)]
    pixels, i, j = max(cells)
    return compass((j + 0.5) * w / 3, (i + 0.5) * h / 3, w, h) if pixels else None

def fact_text(counts, where):
    damage = f"most damage in the {where} of the image" if where else "no damage detected"
    return (f"U-Net analysis: {counts['intact']} intact, {counts['damaged']} damaged and "
            f"{counts['destroyed']} destroyed buildings; {damage}.")

bench_stats, facts, missing = {}, {}, []
for name in tqdm(bench_images, desc="Benchmark images"):
    path = bench_path(name)
    preds = segment(path) if path else None
    if preds is None:
        missing.append(name)
        continue
    pred_std, pred_tta = preds
    bench_stats[name] = image_stats(pred_std, pred_tta)
    cv2.imwrite(str(OUT_DIR / "masks" / f"{Path(name).stem}.png"), pred_tta)
    counts = {CLASS_NAME[c]: count(bench_stats[name], BEST_RULE[c], c) for c in CLASS_NAME}
    where = most_damage_where(pred_tta >= 2)
    facts[name] = {**counts, "damage_where": where, "text": fact_text(counts, where)}

with open(OUT_DIR / "facts.json", "w") as f:
    json.dump(facts, f, indent=1)
print(f"✓ Segmented {len(bench_stats):,} benchmark images ({len(missing)} not found) "
      f"in {(time.time() - t_start) / 60:.1f} min so far")
if missing:
    print(f"⚠ Missing images (their questions count as wrong): {missing[:5]}")
print("Example fact:", next(iter(facts.values()))["text"] if facts else "none")

Benchmark images:   0%|          | 0/2655 [00:00<?, ?it/s]

✓ Segmented 2,653 benchmark images (2 not found) in 29.3 min so far
⚠ Missing images (their questions count as wrong): ['mexico_earthquake_00000140_post_disaster.png', 'spain_volcano_00000031_post_disaster.png']
Example fact: U-Net analysis: 103 intact, 0 damaged and 0 destroyed buildings; no damage detected.


In [8]:
# ── Cell 8: Answer the BDC questions, score them, Gate G4, and the output files ──
rows = []
for e in bdc:
    cls = CLASS_OF[e["cls_description"]]
    stats = bench_stats.get(e["image"])
    n = count(stats, BEST_RULE[cls], cls) if stats is not None else None
    answer = nearest_option(n, e["options_list"]) if n is not None else ""
    rows.append({"id": e["id"], "class": CLASS_NAME[cls], "sensor": e["post_image_type"], "image": e["image"],
                 "truth": e["ground_truth"], "count": n, "answer": answer,
                 "truth_option": e["ground_truth_option"], "correct": answer == e["ground_truth_option"]})
per_q = pd.DataFrame(rows)
per_q.to_csv(OUT_DIR / "H1_v1_per_question.csv", index=False)

def group_row(group, df):
    rel = ((df["count"] - df["truth"]).abs() / df["truth"]).dropna()
    return {"group": group, "questions": len(df), "accuracy_%": round(100 * df["correct"].mean(), 2),
            "median_count_error_%": round(100 * rel.median(), 1) if len(rel) else None}

summary = pd.DataFrame(
    [group_row("all", per_q)]
    + [group_row(f"class {c}", g) for c, g in per_q.groupby("class", sort=False)]
    + [group_row(f"{s} after-image", g) for s, g in per_q.groupby("sensor")]
    + [group_row(f"{s}, {c}", g) for (s, c), g in per_q.groupby(["sensor", "class"], sort=False)])
summary.to_csv(OUT_DIR / "H1_v1_summary.csv", index=False)
print(summary.to_string(index=False))

# Results in the 499A format, so 4_score_results.py can score them next to the VLM runs
res_dir = OUT_DIR / "results" / "building_damage_counting" / "R9-unet-counting"
res_dir.mkdir(parents=True, exist_ok=True)
with open(res_dir / "finished.jsonl", "w") as f:
    for r in rows:
        if r["answer"]:
            f.write(json.dumps({"id": r["id"], "response": r["answer"]}) + "\n")

overall = summary.loc[summary["group"] == "all", "accuracy_%"].item()
optical = summary.loc[summary["group"] == "Optical after-image", "accuracy_%"]
optical = optical.item() if len(optical) else float("nan")
gate = (f"V1 BDC accuracy {overall:.2f}% (optical after-images {optical:.2f}%) against the base model's "
        f"{BASE_BDC}% and our fine-tuned 7B's {FT_BDC}%: "
        + ("the counting half of KR2 is met." if overall > BASE_BDC else "the counting half of KR2 is not met."))
print(f"\nGate G4: {gate}")

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

config = {"task": "H1 V1", "smoke_test": SMOKE_TEST, "model": MODEL_NAME, "checkpoint": str(CKPT),
          "best_rule": {CLASS_NAME[c]: dict(zip(("variant", "closing", "min_area"), r)) for c, r in BEST_RULE.items()},
          "tuning_questions": len(tune_qs), "ground_truth_mask_check": round(gt_acc, 4),
          "benchmark_images": len(bench_stats), "missing_images": len(missing), "bdc_questions": len(per_q),
          "gate_g4": gate, "run_time_min": round((time.time() - t_start) / 60, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# H1 V1: U-Net counting bypass for BDC", "",
    f"{MODEL_NAME} (8-view TTA where tuned), {len(per_q):,} BDC questions{' (SMOKE TEST)' if SMOKE_TEST else ''}. "
    "DisasterM3's BDC answers are 8-connected blob counts of its class masks, so the counter counts blobs in "
    "R9's mask and picks the nearest option.", "",
    "## Result", "", as_markdown(summary), "",
    f"**Gate G4:** {gate}", "",
    "References: base Qwen2.5-VL-7B 34.2%, our fine-tuned 7B 23.58%, the old 499A hybrid 13.11%.", "",
    "## Counter tuned on the validation images", "",
    f"Ground-truth masks answer {gt_acc:.1%} of the {len(tune_qs)} tuning questions.", "",
    as_markdown(best), "",
    "## Notes", "",
    "- SAR questions are answered from the optical twin of the same tile; their truth can differ from the "
    "optical twin's, so they are reported separately.",
    "- R9 never trained on benchmark images. A model trained on D5's benchmark tiles (R12 with all 1,007 "
    "tiles) must not be scored here without excluding those 741 questions.", "",
    "## Files", "",
    "- `H1_v1_summary.csv`, `H1_v1_tuning.csv`, `H1_v1_per_question.csv`, `config.json`",
    "- `results/building_damage_counting/R9-unet-counting/finished.jsonl` (for `4_score_results.py`)",
    "- `facts.json`: per after-image counts, where the damage is, and the fact line for V2",
    "- `masks/<name>.png`: R9 TTA masks (0 background, 1 intact, 2 damaged, 3 destroyed)",
]), encoding="utf-8")
print(f"✓ Wrote the outputs to {OUT_DIR} ({(time.time() - t_start) / 60:.1f} min)")
print("Next: create a Kaggle dataset from this output called h1-v1-facts (used by V2, H2, T2 and T3).")

              group  questions  accuracy_%  median_count_error_%
                all       4982       22.26                  69.2
       class intact       2671       16.66                  58.3
      class damaged       1345       27.36                 100.0
    class destroyed        966       30.64                  89.6
Optical after-image       3583       26.54                  54.5
    SAR after-image       1399       11.29                 100.0
    Optical, intact       1880       21.33                  45.0
        SAR, intact        791        5.56                 100.0
   Optical, damaged        974       31.62                  75.0
       SAR, damaged        371       16.17                 100.0
 Optical, destroyed        729       33.20                  83.3
     SAR, destroyed        237       22.78                 100.0

Gate G4: V1 BDC accuracy 22.26% (optical after-images 26.54%) against the base model's 34.2% and our fine-tuned 7B's 23.58%: the counting half of KR2 is n